# Day 2 — Data Cleaning + SQL Database Design
## Bluestock Mutual Fund Capstone

**Tasks:**
- Clean nav_history, investor_transactions, scheme_performance
- Validate and standardise all 10 CSVs
- Design SQLite star schema (2 dims, 4 facts, 5 reference tables)
- Load all cleaned datasets into SQLite via SQLAlchemy
- Write 10 analytical SQL queries
- Create data dictionary


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sqlalchemy import create_engine, text

PROCESSED_DIR = Path("..") / "data" / "processed"
DB_PATH = Path("..") / "data" / "db" / "bluestock_mf.db"

print("Cleaned files in data/processed/:")
for f in sorted(PROCESSED_DIR.glob("*.csv")):
    df = pd.read_csv(f)
    print(f"  {f.name}: {df.shape}")


## 1. NAV History Cleaning

In [ ]:
nav = pd.read_csv(PROCESSED_DIR / "02_nav_history.csv", parse_dates=["date"])

print("NAV History — Cleaned")
print(f"Shape: {nav.shape}")
print(f"Date range: {nav.date.min()} to {nav.date.max()}")
print(f"Unique funds: {nav.amfi_code.nunique()}")
print(f"Forward-filled rows (weekends/holidays): {nav.is_filled.sum():,}")
print(f"Original trading-day rows: {(~nav.is_filled).sum():,}")
print()
print("Sample:")
print(nav.head(5).to_string(index=False))


## 2. Investor Transactions Cleaning

In [ ]:
tx = pd.read_csv(PROCESSED_DIR / "08_investor_transactions.csv", parse_dates=["transaction_date"])

print("Investor Transactions — Cleaned")
print(f"Shape: {tx.shape}")
print(f"Transaction types: {sorted(tx.transaction_type.unique())}")
print(f"KYC status values: {sorted(tx.kyc_status.unique())}")
print(f"Amount > 0: {(tx.amount_inr > 0).all()}")
print(f"Unique investors: {tx.investor_id.nunique():,}")
print()
print("Transaction type breakdown:")
print(tx.transaction_type.value_counts())


## 3. Scheme Performance Cleaning

In [ ]:
perf = pd.read_csv(PROCESSED_DIR / "07_scheme_performance.csv")

print("Scheme Performance — Cleaned")
print(f"Shape: {perf.shape}")
print(f"Expense ratio range: {perf.expense_ratio_pct.min()} - {perf.expense_ratio_pct.max()}")
return_cols = ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct']
print(f"All returns numeric: {perf[return_cols].dtypes.eq('float64').all()}")
flagged = perf[perf.anomaly_flags != ""]
print(f"Anomaly flags: {len(flagged)} rows flagged")
if len(flagged):
    print(flagged[["scheme_name", "anomaly_flags"]].to_string(index=False))


## 4. SQLite Star Schema

In [ ]:
engine = create_engine(f"sqlite:///{DB_PATH}")

with engine.connect() as conn:
    tables = conn.execute(text(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    )).fetchall()
    print("Tables in bluestock_mf.db:")
    for t in tables:
        count = conn.execute(text(f"SELECT COUNT(*) FROM {t[0]}")).scalar()
        print(f"  {t[0]}: {count:,} rows")


## 5. Sample SQL Queries

In [ ]:
from sqlalchemy import text
import pandas as pd

engine_path = f"sqlite:///{DB_PATH}"
engine2 = create_engine(engine_path)

with engine2.connect() as conn:
    print("Q1: Top 5 funds by AUM")
    print("-" * 50)
    q1 = '''
    SELECT f.scheme_name, f.fund_house, p.aum_crore
    FROM fact_performance p JOIN dim_fund f ON f.amfi_code = p.amfi_code
    ORDER BY p.aum_crore DESC LIMIT 5
'''
    result = pd.read_sql(q1, conn)
    print(result.to_string(index=False))

    print()
    print("Q4: Transactions by state")
    print("-" * 50)
    q4 = '''
    SELECT state, COUNT(*) as num_transactions,
           ROUND(SUM(amount_inr),0) as total_amount
    FROM fact_transactions
    GROUP BY state
    ORDER BY num_transactions DESC
'''
    result2 = pd.read_sql(q4, conn)
    print(result2.to_string(index=False))


## Key Findings

1. nav_history: 46,000 trading-day rows expanded to 64,320 calendar-day rows (18,320 forward-filled)
2. investor_transactions: All 32,778 rows passed validation — amounts > 0, transaction types standardised
3. scheme_performance: 3 liquid funds flagged for extreme Sharpe ratio (expected for low-volatility debt funds)
4. Star schema: 11 tables loaded, all row counts verified matching source CSVs
5. SQL queries: 10 analytical queries covering AUM, NAV trends, SIP growth, geographic distribution, and more
